<span style="font-size: 5em">🦜</span>

# __LangGraph Essentials__
## Building a Tool-Calling Agent with LangGraph + Langfuse

An "agent" is just a graph with a loop: a model node that can decide to call a tool, a tool node that executes it, and a conditional edge that sends control back to the model until it's ready to answer. This notebook builds that loop directly with `StateGraph`
The model is a local Ollama model.

This copy adds **Langfuse observability**: every run is traced (graph steps, model calls, tool calls) via the Langfuse LangChain `CallbackHandler`, so you can inspect it in the Langfuse UI.

## Setup

Load and/or check for needed environmental variables. Langfuse needs `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` and `LANGFUSE_BASE_URL` (see `langfuse/` and `module4/17_LF_observability/.env.example` for the self-hosted instance).

In [ ]:
%%capture --no-stderr
%pip install --quiet -U "langgraph>=1.0.0" langchain_core langchain_ollama "langfuse>=4"
%pip install --upgrade python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

# Ollama server URL - override with OLLAMA_BASE_URL env var
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

# Langfuse (SDK v4 reads LANGFUSE_BASE_URL; defaults to the local self-hosted instance)
os.environ.setdefault("LANGFUSE_BASE_URL", "http://localhost:3000")
os.environ.setdefault("LANGFUSE_TRACING_ENVIRONMENT", "lab")
os.environ.setdefault("LANGFUSE_PUBLIC_KEY", "pk-lf-18873984-84f8-4264-8f05-01c4aa97a39f")
os.environ.setdefault("LANGFUSE_SECRET_KEY", "sk-lf-8f8810fd-b754-44f9-949d-5d55972c43c4")

In [ ]:
from typing import Annotated, Literal, TypedDict

from IPython.display import Image, display
from langchain_core.messages import AnyMessage, ToolMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langfuse import get_client, propagate_attributes
from langfuse.langchain import CallbackHandler
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

## Langfuse tracing

<a id='langfuse_setup'></a>

`get_client()` returns the Langfuse client configured from the environment. `auth_check()` confirms the keys and server URL work before we start.

Tracing is attached per run by passing a `CallbackHandler` in the graph's `config["callbacks"]`. LangGraph propagates it to every node, so the model calls and tool calls are nested under a single trace. Create a **fresh handler for each request** so traces don't bleed into each other.

In [ ]:
langfuse = get_client()

if langfuse.auth_check():
    print("✅ Langfuse client authenticated")
else:
    print("❌ Langfuse auth failed - check LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY / LANGFUSE_BASE_URL")


def traced_config(run_name: str) -> dict:
    """LangGraph run config that sends this run to Langfuse."""
    return {"callbacks": [CallbackHandler()], "run_name": run_name}

## State

<a id='state_definition'></a>

The agent's state is a running list of messages. `add_messages` is a reducer: each node returns new messages, and LangGraph appends them to the existing list rather than overwriting it — this is what gives the agent conversational memory within a single run.

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

## Define a tool

<a id='tool_definition'></a>

A single, simple tool the agent can call.

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    print(f"🌤️ Looking up weather for {city}")
    return f"It's always sunny in {city}!"


tools = [get_weather]
tools_by_name = {t.name: t for t in tools}

## Model node

<a id='model_node'></a>

`bind_tools` tells the model which tools are available so it can request a tool call instead of (or in addition to) answering directly. The node itself is just a function: read the message history from state, call the model, and return the new message.

In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant."

llm = ChatOllama(
    model="llama3.2:latest",
    temperature=0.7,
    base_url=OLLAMA_BASE_URL,
)
llm_with_tools = llm.bind_tools(tools)


def call_model(state: AgentState) -> AgentState:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return AgentState(messages=[response])

## Tool node

<a id='tool_node'></a>

When the model responds with one or more tool calls, this node executes each one and wraps the result in a `ToolMessage` so the model can read it on the next turn.

In [ ]:
def call_tools(state: AgentState) -> AgentState:
    last_message = state["messages"][-1]
    results = []
    for tool_call in last_message.tool_calls:
        tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
        results.append(
            ToolMessage(content=str(tool_result), tool_call_id=tool_call["id"])
        )
    return AgentState(messages=results)

## Conditional routing

<a id='conditional_edge_func'></a>

After the model node runs, this function decides where to go next: back out to `tools` if the model asked for a tool call, or `END` if it produced a final answer.

In [ ]:
def should_continue(state: AgentState) -> Literal["tools", END]:
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END

## Build the graph

<a id='agent_graph'></a>

Wire the nodes together: `model` runs first, `should_continue` decides whether to route to `tools` or stop, and `tools` always loops back to `model` so it can use the tool's result.

In [ ]:
builder = StateGraph(AgentState)

builder.add_node("model", call_model)
builder.add_node("tools", call_tools)

builder.add_edge(START, "model")
builder.add_conditional_edges("model", should_continue, ["tools", END])
builder.add_edge("tools", "model")

graph = builder.compile()

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

## Run the agent

<a id='graph_invoke'></a>

`propagate_attributes` tags the trace with a session, user and tags so you can filter it in Langfuse. The `config=traced_config(...)` argument is what turns tracing on for this run.

In [ ]:
with propagate_attributes(session_id="lab7-tool-agent", user_id="student", tags=["lab7", "tool-calling"]):
    result = graph.invoke(
        {"messages": [{"role": "user", "content": "What's the weather like in Bengaluru?"}]},
        config=traced_config("weather-bengaluru"),
    )
print(result["messages"][-1].content)

Streaming shows each step of the loop: the model deciding to call the tool, the tool node running it, and the model producing the final answer. The same trace is recorded in Langfuse.

In [ ]:
with propagate_attributes(session_id="lab7-tool-agent", user_id="student", tags=["lab7", "streaming"]):
    for step in graph.stream(
        {"messages": [{"role": "user", "content": "What's the weather like in Tokyo?"}]},
        config=traced_config("weather-tokyo-stream"),
        stream_mode="values",
    ):
        step["messages"][-1].pretty_print()